In [1]:
from pathlib import Path
import torch

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

graph_path = ROOT / "data/processed/protein_function_graph_aac_score700.pt"

# weights_only=False is needed because this file contains a PyG Data object.
graph = torch.load(graph_path, weights_only=False)

print(graph)

c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Data(x=[18841, 22], edge_index=[2, 1067636], y=[18841, 96], edge_weight=[1067636], labeled_mask=[18841])


In [2]:
SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

labelled_nodes = torch.where(graph.labeled_mask)[0]

generator = torch.Generator().manual_seed(SEED)
shuffled_nodes = labelled_nodes[
    torch.randperm(len(labelled_nodes), generator=generator)
]

num_labelled = len(shuffled_nodes)
num_train = int(TRAIN_RATIO * num_labelled)
num_validation = int(VALIDATION_RATIO * num_labelled)

train_nodes = shuffled_nodes[:num_train]
validation_nodes = shuffled_nodes[num_train:num_train + num_validation]
test_nodes = shuffled_nodes[num_train + num_validation:]

graph.train_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
graph.validation_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
graph.test_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)

graph.train_mask[train_nodes] = True
graph.validation_mask[validation_nodes] = True
graph.test_mask[test_nodes] = True

print("Labelled proteins:", num_labelled)
print("Train proteins:", int(graph.train_mask.sum()))
print("Validation proteins:", int(graph.validation_mask.sum()))
print("Test proteins:", int(graph.test_mask.sum()))

Labelled proteins: 12020
Train proteins: 8414
Validation proteins: 1803
Test proteins: 1803


In [3]:
# Normalize features using training proteins only.
# Validation/test information is not used here.
train_feature_mean = graph.x[graph.train_mask].mean(dim=0)
train_feature_std = graph.x[graph.train_mask].std(dim=0, unbiased=False)

# Avoid division by zero in any constant feature.
train_feature_std = train_feature_std.clamp_min(1e-8)

graph.x = (graph.x - train_feature_mean) / train_feature_std

print("Train-feature mean after normalization:")
print(graph.x[graph.train_mask].mean(dim=0).round(decimals=4))

print("\nTrain-feature standard deviation after normalization:")
print(graph.x[graph.train_mask].std(dim=0, unbiased=False).round(decimals=4))

Train-feature mean after normalization:
tensor([0., -0., -0., 0., 0., 0., -0., 0., 0., 0., -0., 0., 0., 0., 0., 0., 0., -0., 0., -0., 0., -0.])

Train-feature standard deviation after normalization:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.])


In [4]:
def split_report(split_name, split_mask):
    labels_present = (graph.y[split_mask].sum(dim=0) > 0).sum().item()
    labels_missing = graph.y.shape[1] - labels_present

    print(f"{split_name}")
    print(f"  proteins: {int(split_mask.sum()):,}")
    print(f"  GO labels represented: {labels_present}/{graph.y.shape[1]}")
    print(f"  GO labels missing: {labels_missing}")
    print()

split_report("Train", graph.train_mask)
split_report("Validation", graph.validation_mask)
split_report("Test", graph.test_mask)

Train
  proteins: 8,414
  GO labels represented: 96/96
  GO labels missing: 0

Validation
  proteins: 1,803
  GO labels represented: 96/96
  GO labels missing: 0

Test
  proteins: 1,803
  GO labels represented: 96/96
  GO labels missing: 0



In [5]:
split_graph_path = ROOT / "data/processed/protein_function_graph_aac_score700_split.pt"
scaler_path = ROOT / "data/processed/aac_feature_scaler.pt"

torch.save(graph, split_graph_path)

torch.save(
    {
        "mean": train_feature_mean,
        "std": train_feature_std,
        "seed": SEED,
    },
    scaler_path,
)

print("Saved split graph:", split_graph_path)
print("Saved feature scaler:", scaler_path)

Saved split graph: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\protein_function_graph_aac_score700_split.pt
Saved feature scaler: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\aac_feature_scaler.pt
